[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luckeywolf/colab-torrent-to-drive/blob/main/Torrent_To_Google_Drive.ipynb)


# ⚡ High-Speed Torrent to Google Drive Downloader
Download torrents and magnet links directly into Google Drive with a live, real-time progress dashboard and corruption-free sync.

### 📊 Real-Time Progress Dashboard & Safe Sync
- **Live Speed:** Real-time download transfer rates.
- **Estimated Time (ETA):** Accurate remaining time countdown.
- **Corruption-Free Sync:** Buffers locally on high-speed SSD to prevent Google Drive FUSE file truncation.
- **Direct Sync:** Flushes full verified files into your Google Drive (`/Torrent` folder).

In [ ]:
# ==========================================
# ⚡ High-Speed Torrent Downloader with Safe Drive Sync
# ==========================================
from google.colab import drive
import os
import subprocess
import re
import shutil
import time
from IPython.display import HTML, display

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount("/content/drive")

# 2. Install Aria2
print("Installing Aria2...")
!apt-get update -qq && apt-get install -y aria2 > /dev/null
print("Aria2 ready!")

# 3. Setup paths:
# We buffer to fast local NVMe SSD first to prevent Google Drive FUSE truncation
LOCAL_CACHE_DIR = "/content/download_cache"
DRIVE_SAVE_DIR = "/content/drive/My Drive/Torrent"

os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# 4. Target input
target = input("Enter Magnet link or .torrent URL (or press Enter to upload .torrent file): ").strip()

if not target:
    from google.colab import files
    print("Please upload your .torrent file:")
    uploaded = files.upload()
    if uploaded:
        target = list(uploaded.keys())[0]

def render_progress_card(percent=0.0, speed="0 B", downloaded="0 B", total="0 B", eta="--", peers="0", status="Initializing..."):
    return f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #16181d; border-radius: 14px; padding: 20px; max-width: 620px; box-shadow: 0 10px 25px rgba(0,0,0,0.4); border: 1px solid #2a2d34; color: #f3f4f6; margin: 12px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 14px;">
        <div style="font-weight: 700; font-size: 15px; display: flex; align-items: center; gap: 8px;">
          <span style="display: inline-block; width: 10px; height: 10px; border-radius: 50%; background: #10b981; box-shadow: 0 0 10px #10b981;"></span>
          <span>{status}</span>
        </div>
        <div style="font-size: 13px; color: #9ca3af; background: #202227; padding: 4px 10px; border-radius: 6px; border: 1px solid #2e3138;">
          ETA: <strong style="color: #60a5fa;">{eta}</strong>
        </div>
      </div>

      <div style="background: #252830; border-radius: 8px; height: 14px; overflow: hidden; position: relative; margin-bottom: 16px; border: 1px solid #333640;">
        <div style="background: linear-gradient(90deg, #6366f1, #06b6d4, #10b981); height: 100%; width: {percent}%; border-radius: 8px; transition: width 0.3s ease;"></div>
      </div>

      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; text-align: center; background: #1d1f25; padding: 12px; border-radius: 10px; border: 1px solid #2a2d34;">
        <div>
          <div style="font-size: 11px; text-transform: uppercase; color: #9ca3af; font-weight: 600; letter-spacing: 0.5px;">Progress</div>
          <div style="font-size: 15px; font-weight: 700; color: #38bdf8; margin-top: 2px;">{percent:.1f}%</div>
        </div>
        <div>
          <div style="font-size: 11px; text-transform: uppercase; color: #9ca3af; font-weight: 600; letter-spacing: 0.5px;">Speed</div>
          <div style="font-size: 15px; font-weight: 700; color: #34d399; margin-top: 2px;">{speed}/s</div>
        </div>
        <div>
          <div style="font-size: 11px; text-transform: uppercase; color: #9ca3af; font-weight: 600; letter-spacing: 0.5px;">Downloaded</div>
          <div style="font-size: 13px; font-weight: 600; color: #e5e7eb; margin-top: 2px;">{downloaded} / {total}</div>
        </div>
        <div>
          <div style="font-size: 11px; text-transform: uppercase; color: #9ca3af; font-weight: 600; letter-spacing: 0.5px;">Peers</div>
          <div style="font-size: 15px; font-weight: 700; color: #c084fc; margin-top: 2px;">{peers}</div>
        </div>
      </div>
    </div>
    """

def render_syncing_card():
    return """
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #1c1d16; border-radius: 14px; padding: 22px; max-width: 620px; box-shadow: 0 10px 25px rgba(0,0,0,0.4); border: 1px solid #d97706; color: #f3f4f6; margin: 12px 0;">
      <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 8px;">
        <div style="background: #f59e0b; border-radius: 50%; width: 28px; height: 28px; display: flex; align-items: center; justify-content: center; font-size: 16px; font-weight: bold; color: black;">⏳</div>
        <div style="font-weight: 700; font-size: 16px; color: #fbbf24;">Download Complete! Transferring to Google Drive...</div>
      </div>
      <div style="font-size: 13px; color: #d1d5db; line-height: 1.5;">
        Writing the full verified file sequentially to Google Drive to ensure 0% corruption and full file size. Please wait a moment...
      </div>
    </div>
    """

def render_completed_card(save_path, total_size_str):
    return f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; background: #131d17; border-radius: 14px; padding: 22px; max-width: 620px; box-shadow: 0 10px 25px rgba(0,0,0,0.4); border: 1px solid #059669; color: #f3f4f6; margin: 12px 0;">
      <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 10px;">
        <div style="background: #10b981; border-radius: 50%; width: 28px; height: 28px; display: flex; align-items: center; justify-content: center; font-size: 16px; font-weight: bold; color: white;">✓</div>
        <div style="font-weight: 700; font-size: 16px; color: #34d399;">Fully Synced & Ready ({total_size_str})!</div>
      </div>
      <div style="font-size: 13px; color: #d1d5db; line-height: 1.5;">
        All files have been verified, fully synced, and flushed to Google Drive at:
        <div style="margin-top: 8px; padding: 10px 12px; background: #1b2820; border-radius: 6px; font-family: monospace; font-size: 12px; color: #6ee7b7; word-break: break-all; border: 1px solid #047857;">
          {save_path}
        </div>
      </div>
    </div>
    """

def parse_aria2_progress(line):
    m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)\]", line)
    if not m_size:
        m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)", line)
    if not m_size:
        return None
    downloaded, total, percent = m_size.groups()
    m_cn = re.search(r"CN:(\d+)", line)
    cn = m_cn.group(1) if m_cn else "0"
    m_dl = re.search(r"DL:([0-9.]+[KMGT]?i?B)", line)
    dl = m_dl.group(1) if m_dl else "0B"
    m_eta = re.search(r"ETA:([0-9A-Za-z]+)", line)
    eta = m_eta.group(1) if m_eta else "--"
    return {
        "downloaded": downloaded,
        "total": total,
        "percent": float(percent),
        "peers": cn,
        "speed": dl,
        "eta": eta
    }

if target:
    cmd = [
        "aria2c",
        "-c",
        "--enable-dht=true",
        "--bt-enable-lpd=true",
        "--max-connection-per-server=16",
        "--split=16",
        "--seed-time=0",
        "--summary-interval=1",
        f"--dir={LOCAL_CACHE_DIR}",
        target
    ]

    display_handle = display(HTML(render_progress_card(status="Connecting to peers...")), display_id=True)
    
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    buffer = ""
    while True:
        char = proc.stdout.read(1)
        if not char and proc.poll() is not None:
            break
        if char in ("\r", "\n"):
            line = buffer.strip()
            buffer = ""
            if line:
                data = parse_aria2_progress(line)
                if data:
                    display_handle.update(HTML(render_progress_card(
                        percent=data["percent"],
                        speed=data["speed"],
                        downloaded=data["downloaded"],
                        total=data["total"],
                        eta=data["eta"],
                        peers=data["peers"],
                        status="Downloading..."
                    )))
                elif "downloading metadata" in line.lower():
                    display_handle.update(HTML(render_progress_card(status="Fetching torrent metadata...")))
        else:
            buffer += char

    if proc.returncode == 0:
        display_handle.update(HTML(render_syncing_card()))
        
        total_bytes = 0
        for item in os.listdir(LOCAL_CACHE_DIR):
            src = os.path.join(LOCAL_CACHE_DIR, item)
            dst = os.path.join(DRIVE_SAVE_DIR, item)
            if os.path.isdir(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
                for root, dirs, files in os.walk(src):
                    for f in files:
                        total_bytes += os.path.getsize(os.path.join(root, f))
            else:
                shutil.copy2(src, dst)
                total_bytes += os.path.getsize(src)
        
        os.sync()
        
        size_mb = total_bytes / (1024 * 1024)
        size_str = f"{size_mb / 1024:.2f} GB" if size_mb >= 1024 else f"{size_mb:.1f} MB"
        
        display_handle.update(HTML(render_completed_card(DRIVE_SAVE_DIR, size_str)))
    else:
        print("Aria2 exited with error code:", proc.returncode)
else:
    print("No link or file provided.")

